# Notebook19 prospective development protocol
Purpose: freeze two fresh paired seeds and a single local-weight ablation before pilot training; summarize DEV evidence without statistical confirmation. Inputs: source-bound plan and pinned TRAIN/DEV membership. Outputs: immutable pilot registration and descriptive result. Runtime seconds; no training/checkpoints in these definition cells. Main Notebook19 owns the original three-hour clock and exact resume.

In [ ]:
from pathlib import Path
import json,nbformat,types,math,statistics,time
_SELF=Path(globals().get('__file__','/home/benjamin/workspace/ahr_go/fmm_jeppa/experiments/19_compute_and_gate/protocol.ipynb'))
_space={'__file__':str(_SELF.with_name('common.ipynb'))}
for _cell in nbformat.read(_SELF.with_name('common.ipynb'),4).cells:
    if _cell.cell_type=='code'and'ahr-export'in _cell.metadata.get('tags',[]):exec(compile(_cell.source,str(_SELF),'exec'),_space)
C=type('Common',(),_space)


In [ ]:
VARIANTS=('vit','learned_optimized','learned_global_prior')
SEEDS=(20262201,20262202)
def prior_seeds(root,own_artifact):
    used=set();own=(Path(own_artifact)/'pilot_registration.json').resolve()
    for parent in ('notebook_16','notebook_17','notebook_18','notebook_19'):
        for pattern in ('confirmation_registration.json','pilot_registration.json'):
            for p in(Path(root)/'artifacts'/parent).rglob(pattern):
                if p.resolve()!=own:used.update(map(int,C.read(p).get('seeds',[])))
    return used

def register(root,artifact,plan,bundle):
    artifact=Path(artifact);seeds=list(plan['registered_seeds'])
    if seeds!=list(SEEDS)or list(plan['variants'])!=list(VARIANTS):raise ValueError('Fixed two-pair three-condition pilot required')
    if prior_seeds(root,artifact)&set(seeds):raise ValueError('Fresh pilot seeds were already reserved')
    if plan['updates']!=4000 or plan['selected']['rank']!=64 or plan['selected']['local_radius']!=1 or plan['active_wall_seconds_per_arm']!=900 or plan['matched_updates']!=200:raise ValueError('Predeclared wall/update endpoints changed')
    for seed in seeds:
        candidates={a['variant']:a['config']['learned_kernel19']for a in plan['arms']if a['seed']==seed and a['variant']!='vit'}
        if set(candidates)!=set(VARIANTS[1:]):raise ValueError('Both learned conditions required')
        if candidates['learned_optimized']['gate_initial']!=.5 or candidates['learned_global_prior']['gate_initial']!=.1:raise ValueError('Registered local initialization changed')
        a={k:v for k,v in candidates['learned_optimized'].items()if k!='gate_initial'};b={k:v for k,v in candidates['learned_global_prior'].items()if k!='gate_initial'}
        if a!=b:raise ValueError('Additional mathematical factor changed')
    train=set(bundle['train_ids']);probe=set(bundle['probe_train_ids']);dev=set(bundle['dev_ids']);held=set(bundle['heldout_ids'])
    if train&dev or probe&dev or (train|probe|dev)&held:raise ValueError('TRAIN/DEV/official membership overlap')
    path=artifact/'pilot_registration.json';old=C.read(path)if path.exists()else None
    value={'schema':'notebook19-development-pilot/v1','run_id':plan['run_id'],'seeds':seeds,'variants':list(VARIANTS),
        'created_unix':old['created_unix']if old else time.time(),'plan_hash':C.digest(plan),'source_hashes':plan['source_hashes'],
        'active_seconds_per_arm':900,'matched_updates':200,'initial_control':True,'scheduler_horizon':4000,
        'only_mathematical_change':'Initial local mixture probability0.1 versus0.5; rank64 and all other options identical',
        'engineering_change':'Certified unchanged-mathematics topology cache and deterministic transpose-gather backward',
        'metrics':['Certified DEV linear top1','DEV20NN accuracy','own-initial gain','actual updates and wall time','allocated memory'],
        'membership':{k:C.digest(list(bundle[k]))for k in('train_ids','probe_train_ids','dev_ids','heldout_ids')},
        'official_test':'CLOSED','official_predictive_pixels_read':0,'sample_size':2,
        'scope':'Exploratory mechanisms and efficiency; no significance, superiority, equivalence or global scaling claim',
        'selection':'Fixed200 and last completed equal-wall checkpoint; no best-DEV checkpoint selection',
        'favorable_signal_rule':'Both paired final DEV linear effects positive, mean>=0.5pp and mean paired kNN effect>=0; this is a screening rule, not statistical confirmation',
        'next_confirmation':'Any later confirmation needs fresh paired seeds and an unopened official subset; current exposed official images cannot select variants',
        'unknown_policy':'Missing matched checkpoint/readout or kNN remainsUNKNOWN; no invented zero or silent budget shortening'}
    return C.immutable(path,value)

def analyze(plan,evaluation,training,profiling=None):
    endpoints={};all_primary=True
    for name,key in [('initial','initial_rows'),('matched','matched_rows'),('final','rows')]:
        rows=evaluation.get(key,[]);pairs=[];missing=[]
        for seed in plan['registered_seeds']:
            group={}
            for row in rows:
                if int(row['seed'])==int(seed):
                    if row['variant']in group:raise ValueError('Duplicate paired readout')
                    group[row['variant']]=row
            if set(group)-set(VARIANTS):raise ValueError('Unregistered condition')
            for candidate in VARIANTS[1:]:
                base=group.get('vit',{});other=group.get(candidate,{})
                valid=all(x.get('status')=='COMPLETE'and x.get('converged')and x.get('numerical_valid')and isinstance(x.get('accuracy_percent'),(int,float))and math.isfinite(x['accuracy_percent'])and 0<=x['accuracy_percent']<=100 and not x.get('fixture_source')for x in(base,other))
                if not valid:missing.append({'seed':seed,'candidate':candidate,'status':'UNKNOWN'});continue
                knn_valid=all(isinstance(x.get('knn_accuracy_percent'),(int,float))and math.isfinite(x['knn_accuracy_percent'])for x in(base,other))
                pairs.append({'seed':seed,'candidate':candidate,'vit_percent':base['accuracy_percent'],'candidate_percent':other['accuracy_percent'],
                    'delta_pp':other['accuracy_percent']-base['accuracy_percent'],'knn_delta_pp':other['knn_accuracy_percent']-base['knn_accuracy_percent']if knn_valid else None})
        summaries={}
        for candidate in VARIANTS[1:]:
            subset=[x for x in pairs if x['candidate']==candidate];knn=[x['knn_delta_pp']for x in subset if x['knn_delta_pp']is not None]
            summaries[candidate]={'status':'COMPLETE'if len(subset)==len(SEEDS)else'UNKNOWN','n_pairs':len(subset),
                'mean_delta_pp':statistics.mean(x['delta_pp']for x in subset)if subset else None,
                'observed_delta_range_pp':[min(x['delta_pp']for x in subset),max(x['delta_pp']for x in subset)]if subset else None,
                'mean_knn_delta_pp':statistics.mean(knn)if len(knn)==len(SEEDS)else None,
                'paired_results':subset,'scope':'Observed two-pair DEV effects; no confidence interval or pvalue'}
        endpoints[name]={'status':'UNKNOWN'if missing else'COMPLETE','comparisons':summaries,'missing':missing}
        if name!='matched'and missing:all_primary=False
    signals={}
    for candidate,s in endpoints['final']['comparisons'].items():
        favorable=s['status']=='COMPLETE'and s['mean_delta_pp']>=.5 and all(x['delta_pp']>0 for x in s['paired_results'])and s['mean_knn_delta_pp']is not None and s['mean_knn_delta_pp']>=0
        signals[candidate]='FAVORABLE_EXPLORATORY_SIGNAL'if favorable else'NO_COMPLETE_FAVORABLE_SIGNAL'
    return {'schema':'notebook19-development-result/v1','status':'COMPLETE'if all_primary and training.get('status')=='COMPLETE'and training.get('fair_wall_budget_verified',False)else'UNKNOWN',
        'run_id':plan['run_id'],'phase':'finished','seeds':plan['registered_seeds'],'variants':list(VARIANTS),'endpoints':endpoints,
        'signals':signals,'official_test':'CLOSED','official_test_pixels_read':0,'matched_status':endpoints['matched']['status'],
        'inference':'Descriptive pilot only; no statistical significance or superiority claim','additional_online_parameters':98328,
        'profiling_status':(profiling or{}).get('status'),'source_hashes':plan['source_hashes'],'plan_hash':C.digest(plan)}
